Credit Card Fraud Detection and Transaction Risk Analytics

2. Data Cleaning and Preparation
This notebook prepares the raw credit card transaction dataset for exploratory analysis, SQL analysis, Power BI reporting, and predictive modelling.

The cleaning process will preserve the original source data and document each transformation. Particular attention will be given to missing values, data types, categorical consistency, binary indicators, unusual numerical values, and potential target leakage.

In [10]:
import pandas as pd
import numpy as np

In [11]:
file_path = "../data/raw/credit_card_fraud_detection_2026.csv"

raw_df = pd.read_csv(file_path)

raw_df.shape

(100000, 43)

In [12]:
clean_df = raw_df.copy()

In [13]:
clean_df['transaction_datetime'] = pd.to_datetime(
    clean_df['transaction_datetime']
)

clean_df['transaction_datetime'].dtype

dtype('<M8[us]')

In [14]:
clean_df['transaction_datetime'].isna().sum()

np.int64(0)

In [15]:
clean_df['browser'] = clean_df['browser'].fillna('Not Applicable')

In [16]:
clean_df['browser'].value_counts(dropna=False)

browser
Not Applicable    41709
Chrome            20359
Safari            12841
App               12759
Firefox            5448
Edge               4544
Other              2340
Name: count, dtype: int64

In [17]:
clean_df['browser'].isna().sum()

np.int64(0)

Missing Browser Values

Missing browser values were associated exclusively with ATM and In-store transactions.

Because browser information is not applicable to these transaction channels in the available dataset, the missing values were labelled Not Applicable rather than deleting the affected records or imputing a specific browser.

This preserves all transaction records while explicitly distinguishing structurally unavailable browser information from observed browser categories.

In [18]:
binary_columns = [
    'card_present',
    'international_transaction',
    'high_risk_country',
    'used_chip',
    'used_pin',
    'is_weekend',
    'is_fraud'
]

for col in binary_columns:
    print(f"{col}: {sorted(clean_df[col].unique())}")

card_present: [np.int64(0), np.int64(1)]
international_transaction: [np.int64(0), np.int64(1)]
high_risk_country: [np.int64(0), np.int64(1)]
used_chip: [np.int64(0), np.int64(1)]
used_pin: [np.int64(0), np.int64(1)]
is_weekend: [np.int64(0), np.int64(1)]
is_fraud: [np.int64(0), np.int64(1)]


In [19]:
categorical_columns = [
    'merchant_category',
    'merchant_country',
    'transaction_day_of_week',
    'customer_gender',
    'customer_state',
    'transaction_channel',
    'device_type',
    'browser',
    'email_domain',
    'fraud_type'
]

for col in categorical_columns:
    print(f"\n{col}")
    print(sorted(clean_df[col].astype(str).unique()))


merchant_category
['Auto & Transportation', 'Clothing & Apparel', 'Education', 'Electronics & Technology', 'Entertainment & Streaming', 'Financial Services', 'Gas Station', 'Grocery & Supermarket', 'Healthcare & Pharmacy', 'Home Improvement', 'Hotel & Lodging', 'Online Retail', 'Restaurant & Dining', 'Travel & Airlines', 'Utilities & Telecom']

merchant_country
['Australia', 'Brazil', 'Canada', 'China', 'France', 'Germany', 'India', 'Italy', 'Japan', 'Mexico', 'Netherlands', 'Nigeria', 'Romania', 'Russia', 'South Africa', 'South Korea', 'Spain', 'UAE', 'UK', 'USA']

transaction_day_of_week
['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']

customer_gender
['Female', 'Male', 'Other']

customer_state
['AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'MI', 'MN', 'MO', 'MS', 'NC', 'NE', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'SC', 'TN', 'TX', 'UT', 'VA', 'WA', 'WI', 'WV']

transaction_channel


Binary and Categorical Validation

All reviewed binary indicator variables contain only the expected values 0 and 1.

Categorical variables were also inspected for inconsistent capitalization, spelling, unexpected labels, and formatting differences. No obvious inconsistencies requiring correction were identified during the initial validation.

Therefore, the existing categorical labels were retained.

In [20]:
range_columns = [
    'transaction_amount',
    'transaction_hour',
    'customer_age',
    'account_age_days',
    'credit_limit',
    'credit_score',
    'num_cards_held',
    'days_since_last_transaction',
    'distance_from_home_km',
    'distance_from_last_tx_km',
    'amount_vs_avg_ratio',
    'fraud_amount_lost'
]

clean_df[range_columns].agg(['min', 'max']).T

,min,max
transaction_amount,0.500,15000.000
transaction_hour,0.000,23.000
customer_age,18.000,85.000
account_age_days,1.000,3650.000
credit_limit,500.000,36670.000
credit_score,328.000,850.000
num_cards_held,1.000,5.000
days_since_last_transaction,0.000,47.300
distance_from_home_km,0.000,14999.800
distance_from_last_tx_km,0.000,11999.600


Numerical Range Validation

Selected numerical variables were reviewed using their minimum and maximum values.

No obvious invalid negative values were identified. Transaction hour ranges from 0 to 23, customer age ranges from 18 to 85, and the reviewed count and monetary variables contain non-negative values.

Several variables contain large maximum values, particularly geographic distance and transaction amount relative to average spending. These observations will not be automatically removed or capped because unusual behaviour may contain meaningful fraud-related information.

Their distributions will be investigated during exploratory data analysis before any outlier-treatment decision is made.

In [21]:
pd.crosstab(
    clean_df['is_fraud'],
    clean_df['fraud_type']
)

fraud_type,Account-Takeover,Card-Not-Present,Card-Present,Legitimate,Synthetic-Identity
is_fraud,,,,,
0,0,0,0,97945,0
1,447,1054,245,0,309


In [22]:
clean_df.groupby('is_fraud')['fraud_amount_lost'].agg(
    ['count', 'min', 'max', 'mean']
).round(2)

,count,min,max,mean
is_fraud,,,,
0,97945,0.00,0.00,0.00
1,2055,0.58,4482.16,136.18


Target Leakage Assessment

The variables fraud_type and fraud_amount_lost contain information that directly reveals the fraud outcome.

All legitimate transactions are labelled Legitimate in fraud_type, while every transaction belonging to a fraud-specific category is labelled as fraudulent.

Similarly, all legitimate transactions have a fraud_amount_lost value of zero, while all fraudulent transactions have a positive recorded loss.

These variables will therefore be retained for descriptive analysis, financial-impact reporting, SQL analysis, and Power BI reporting, but excluded from the predictive feature set used to predict is_fraud.

Including these variables as model predictors would introduce target leakage and produce unrealistically optimistic model performance.

In [23]:
clean_df['transaction_date'] = clean_df['transaction_datetime'].dt.date
clean_df['transaction_month'] = clean_df['transaction_datetime'].dt.month
clean_df['transaction_month_name'] = clean_df['transaction_datetime'].dt.month_name()
clean_df['transaction_day'] = clean_df['transaction_datetime'].dt.day

clean_df[
    [
        'transaction_datetime',
        'transaction_date',
        'transaction_month',
        'transaction_month_name',
        'transaction_day',
        'transaction_hour'
    ]
].head()

,transaction_datetime,transaction_date,transaction_month,transaction_month_name,transaction_day,transaction_hour
0,2026-01-01 02:28:51,2026-01-01,1,January,1,2
1,2026-01-01 03:15:18,2026-01-01,1,January,1,3
2,2026-01-01 03:44:38,2026-01-01,1,January,1,3
3,2026-01-01 03:53:41,2026-01-01,1,January,1,3
4,2026-01-01 04:29:08,2026-01-01,1,January,1,4


In [24]:
clean_df.shape

(100000, 47)

In [25]:
print("Rows:", clean_df.shape[0])
print("Columns:", clean_df.shape[1])
print("Missing values:", clean_df.isna().sum().sum())
print("Duplicate rows:", clean_df.duplicated().sum())
print("Duplicate transaction IDs:", clean_df['transaction_id'].duplicated().sum())

Rows: 100000
Columns: 47
Missing values: 0
Duplicate rows: 0
Duplicate transaction IDs: 0


In [26]:
clean_df['is_fraud'].value_counts()

is_fraud
0    97945
1     2055
Name: count, dtype: int64

In [27]:
clean_df.to_csv(
    "../data/processed/credit_card_fraud_clean.csv",
    index=False
)

In [28]:
import os

os.path.exists("../data/processed/credit_card_fraud_clean.csv")

True

Cleaning Summary

The raw dataset was prepared for subsequent exploratory analysis, SQL analysis, Power BI reporting, and predictive modelling.

The following preparation steps were completed:

A separate working copy was created to preserve the raw source data.
transaction_datetime was converted to a datetime data type and validated successfully.
Structurally missing browser values associated with ATM and In-store transactions were labelled Not Applicable.
Binary variables were validated to contain only the expected 0 and 1 values.
Categorical variables were inspected for obvious inconsistencies and retained where no correction was required.
Numerical ranges were reviewed without automatically removing extreme observations that may contain meaningful fraud information.
fraud_type and fraud_amount_lost were identified as target-leakage variables and will be excluded from predictive features while remaining available for descriptive and reporting analysis.
Additional date fields were derived to support temporal analysis, SQL queries, and Power BI reporting.
No transaction records were intentionally removed during the cleaning process.
The cleaned dataset was exported as an analysis-ready dataset while the original raw dataset remained unchanged.